# Generative AI: Assignment 1
## Part 1: Topic Detection and Summarization of News Articles (45 marks)

We use the BBC News Archive dataset (`bbc-news-data.csv`) and a LangChain-powered pipeline to perform: 1) Topic Classification, 2) Summarization, and 3) Key Entity Extraction for each article.

## Step 1: Load the Dataset

In [1]:
import os, json, re, getpass
import pandas as pd
from dotenv import load_dotenv

load_dotenv(r"C:\Dhiren\Jio Institute\Course\Term 4\Gen AI\Codes\GenAI\.env", override=True)

True

In [2]:
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = "not-needed-for-ollama"  # not used by the Ollama backend below

In [3]:
if os.environ["GROQ_API_KEY"]:
    print(f"Groq API Key exists and begins {os.environ['GROQ_API_KEY'][:4]}")
else:
    print("Groq API Key not set (and this is optional)")

Groq API Key exists and begins gsk_


In [4]:
# BBC News Archive dataset is tab-separated with columns: category, filename, title, content
df = pd.read_csv("bbc-news-data.csv", sep="\t")
print(df.shape)
df.head()

(2225, 4)


   category  ...                                            content
0  business  ...   Quarterly profits at US media giant TimeWarne...
1  business  ...   The dollar has hit its highest level against ...
2  business  ...   The owners of embattled Russian oil giant Yuk...
3  business  ...   British Airways has blamed high fuel prices f...
4  business  ...   Shares in UK drinks and food firm Allied Dome...

[5 rows x 4 columns]

In [5]:
# Limit the data to the first 30 articles for this assignment
NUM_ARTICLES = 30
df_subset = df.head(NUM_ARTICLES).reset_index(drop=True)
df_subset.insert(0, "Article_ID", df_subset.index)
df_subset.head()

   Article_ID  ...                                            content
0           0  ...   Quarterly profits at US media giant TimeWarne...
1           1  ...   The dollar has hit its highest level against ...
2           2  ...   The owners of embattled Russian oil giant Yuk...
3           3  ...   British Airways has blamed high fuel prices f...
4           4  ...   Shares in UK drinks and food firm Allied Dome...

[5 rows x 5 columns]

## Step 2: Define the Topic Classification Task (10 marks)

In [6]:
#Using LangChain
from langchain.chat_models import init_chat_model

# Note: running via Ollama (local) instead of Groq for this run, since the Groq
# account's on_demand daily token quota (200,000 tokens/day) was exhausted.
# Swap model_name/model_provider back to ("openai/gpt-oss-120b", "groq") once quota resets.
model_name = "openai/gpt-oss-120b"
llm = init_chat_model(model_name, model_provider="groq", temperature=0.0)

In [7]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

classification_template = """Analyze the following news article and identify its topic as one of the following categories: Business, Entertainment, Politics, Sport, or Tech.

Here are a couple of examples to guide you:
Article: "The government today passed a new bill regarding healthcare funding after weeks of debate in parliament."
Category: Politics

Article: "The home team secured a dramatic win in extra time during last night's cup match."
Category: Sport

Now classify the article below. Only return the single category label, nothing else.

Article:
{article_text}

Category:"""

classification_prompt = ChatPromptTemplate.from_template(classification_template)
classification_chain = classification_prompt | llm | StrOutputParser()

In [8]:
#Show this works for a sample datapoint
sample_article = df_subset.iloc[0]
sample_topic = classification_chain.invoke({"article_text": sample_article["content"]})
print("Title:", sample_article["title"])
print("Actual category (from dataset):", sample_article["category"])
print("Predicted topic:", sample_topic.strip())

Title: Ad sales boost Time Warner profit
Actual category (from dataset): business
Predicted topic: Business


## Step 3: Define the Summarization Task (10 marks)

In [9]:
summary_template = """Summarize the main points of the following news article in 2-3 sentences.
Ensure the summary captures the essence of the article (who/what/when/where/why, as applicable) without personal commentary.

Article:
{article_text}

Summary:"""

summary_prompt = ChatPromptTemplate.from_template(summary_template)
summary_chain = summary_prompt | llm | StrOutputParser()

In [10]:
#Show this works for a sample datapoint
sample_summary = summary_chain.invoke({"article_text": sample_article["content"]})
print(sample_summary)

TimeWarner’s quarterly profit surged 76% to $1.13 billion for the three months to December, driven by higher high‑speed internet sales, stronger advertising revenue and one‑off gains that offset a dip at Warner Bros and a loss of AOL subscribers. The company, now holding an 8% stake in Google, posted a 2% rise in Q4 sales to $11.1 billion, announced plans to restate its 2000 and 2003 results after an SEC probe, and projected about 5% operating‑earnings growth for 2005 despite a 27% slump in its film‑division profit.


## Step 4: Key Entity Extraction (10 marks)

In [11]:
from pydantic import BaseModel, Field

class KeyEntities(BaseModel):
    """From a news article, extract the key entities mentioned - important people, organizations, and locations."""
    entities: list[str] = Field(description="A flat list of important people, organizations, and locations mentioned in the article, empty list if none")

In [12]:
entity_extractor = llm.with_structured_output(KeyEntities)

entity_prompt_text = (
    "From the article below, list the names of any important people, organizations, or places mentioned.\n\n"
    f"Article:\n{sample_article['content']}"
)

In [13]:
#Show this works for a sample datapoint
sample_entities = entity_extractor.invoke(entity_prompt_text)
sample_entities

KeyEntities(entities=['TimeWarner', 'Google', 'Warner Bros', 'AOL', 'US Securities Exchange Commission', 'SEC', 'Lord of the Rings', 'Richard Parsons', 'Bertelsmann'])

## Step 5: Update the DataFrame with Results (15 marks)

We combine classification, summarization, and entity extraction into a single structured-output schema so that only one LLM call is needed per article, then loop over the DataFrame.

In [14]:
class ArticleAnalysis(BaseModel):
    """Analyze a news article: classify its topic, summarize it, and extract key entities."""
    topic: str = Field(description="The single most appropriate topic for the article. Must be exactly one of: Business, Entertainment, Politics, Sport, Tech.")
    summary: str = Field(description="A 2-3 sentence summary of the article capturing who/what/when/where/why as applicable, without personal commentary")
    entities: list[str] = Field(description="A flat list of important people, organizations, and locations mentioned in the article, empty list if none")

analysis_extractor = llm.with_structured_output(ArticleAnalysis)

In [15]:
results = []

for index, row in df_subset.iterrows():
    try:
        article_prompt = f"Article Title: {row['title']}\n\nArticle:\n{row['content']}"
        analysis = analysis_extractor.invoke(article_prompt)
        results.append(analysis.model_dump())
    except Exception as e:
        print(f"Failed at row {index}: {e}")
        results.append({"topic": None, "summary": None, "entities": None})

Failed at row 8: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01kmq405y6ejzrpfg7qa3q7z39` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 7558, Requested 906. Please try again in 3.48s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
Failed at row 9: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01kmq405y6ejzrpfg7qa3q7z39` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 7648, Requested 790. Please try again in 3.285s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
Failed at row 13: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01kmq405y6ejzrpfg7qa3q7z39` servic

In [16]:
results_df = pd.DataFrame(results)
results_df.rename(columns={
    "topic": "Detected_Topic",
    "summary": "Summary",
    "entities": "Key_Entities"
}, inplace=True)
results_df.head()

  Detected_Topic  ...                                       Key_Entities
0       Business  ...  [Time Warner, Google, AOL, Warner Bros, Richar...
1       Business  ...  [Alan Greenspan, Robert Sinche, Bank of Americ...
2       Business  ...  [Yukos, Rosneft, Yugansk, Menatep Group, Tim O...
3       Business  ...  [British Airways, Rod Eddington, Mike Powell, ...
4       Business  ...  [Allied Domecq, Pernod Ricard, Wall Street Jou...

[5 rows x 3 columns]

In [17]:
#Final Pandas dataframe with all original and new columns together
df_final_part1 = pd.concat([df_subset.reset_index(drop=True), results_df], axis=1)
df_final_part1

    Article_ID  ...                                       Key_Entities
0            0  ...  [Time Warner, Google, AOL, Warner Bros, Richar...
1            1  ...  [Alan Greenspan, Robert Sinche, Bank of Americ...
2            2  ...  [Yukos, Rosneft, Yugansk, Menatep Group, Tim O...
3            3  ...  [British Airways, Rod Eddington, Mike Powell, ...
4            4  ...  [Allied Domecq, Pernod Ricard, Wall Street Jou...
5            5  ...  [Japan, Heizo Takenaka, Paul Sheard, Lehman Br...
6            6  ...  [President Bush, Labor Department, BMO Financi...
7            7  ...  [India, G7, London, Palaniappan Chidambaram, U...
8            8  ...                                               None
9            9  ...                                               None
10          10  ...           [Ask Jeeves, Google, Yahoo, Doubleclick]
11          11  ...  [Indonesia, President Susilo Bambang Yudhoyono...
12          12  ...  [Mitsubishi Motors, Peugeot, Takashi Nishioka,...
13    